# 02 — SQL Business Analytics

**Part A extension:** execute the seven SQL business questions in `sql/queries.sql` against the project's SQLite database. This notebook explains database connection, SQL execution, result inspection and checkpoint validation.

In [1]:
from pathlib import Path
import os, sys

# Find the project root whether this notebook is opened from the project folder or notebooks/.
CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Instamart project folder (or its notebooks folder).")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: d:\New folder\Instamart capstone project


In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = PROJECT_ROOT / "data" / "instamart.db"
SQL_PATH = PROJECT_ROOT / "sql" / "queries.sql"
assert DB_PATH.exists(), "Run Notebook 01 first to create data/instamart.db"
assert SQL_PATH.exists(), "sql/queries.sql is missing"
conn = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH)

Connected to: d:\New folder\Instamart capstone project\data\instamart.db


## 1. Inspect database tables

In [3]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
display(tables)
for table in tables["name"]:
    print("\n", table)
    display(pd.read_sql_query(f'SELECT * FROM "{table}" LIMIT 5', conn))

,name
0,customers
1,orders
2,products
3,tickets



 customers


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female
3,C0004,Bengaluru,2024-06-21,24,female
4,C0005,Bengaluru,2024-10-01,25,female



 orders


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
1,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
2,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN
3,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0
4,ORD-108115,C0776,P012,2,2025-04-01 08:02:58,completed,10,104.40,21.0



 products


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245
3,P004,Sunflower Oil 1L,Grocery,155
4,P005,Sugar 1kg,Grocery,48



 tickets


,ticket_id,customer_id,order_id,text,true_category,clean_text
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud,There is an unauthorized transaction on my car...
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing,The invoice for [ORDER_ID] shows a wrong amoun...
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical,The payment page shows an error and the app fr...
3,T0159,C1172,NaN,NaN,none,NaN
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud,I was charged for order [ORDER_ID] that I neve...


## 2. Read and split the SQL script
The project separates its seven queries with the marker `-- BREAK_QUERY --`. We preserve those original queries instead of rewriting them.

In [4]:
sql_text = SQL_PATH.read_text(encoding="utf-8")
query_blocks = [q.strip() for q in sql_text.split("-- BREAK_QUERY --") if q.strip()]
print("Number of SQL queries:", len(query_blocks))
for i, query in enumerate(query_blocks, start=1):
    print(f"\n--- Query {i} ---\n{query}")

Number of SQL queries: 1

--- Query 1 ---
-- Q1: Completed orders with spend > 500, customer total > 8000
SELECT customer_id, ROUND(SUM(order_amount), 2) AS total_spend
FROM orders
WHERE status = 'completed' AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000
ORDER BY total_spend DESC;

-- Q2: Revenue by city and category
SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.city, p.category
ORDER BY c.city ASC, revenue DESC;

-- Q3: Products above average catalogue price
SELECT product_id, product_name, category, ROUND(price_inr, 2) AS price_inr
FROM products
WHERE price_inr > (SELECT AVG(price_inr) FROM products)
ORDER BY price_inr DESC;

-- Q4: Customers with zero completed orders
SELECT c.customer_id, c.city, c.signup_date
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id AND o

## 3. Execute each query and inspect results

In [5]:
sql_results = {}

for i, query in enumerate(query_blocks, start=1):

    queries = [
        q.strip()
        for q in query.split(';')
        if q.strip()
    ]

    for j, single_query in enumerate(queries, start=1):

        result = pd.read_sql_query(single_query, conn)

        key = f"Q{i}_{j}"
        sql_results[key] = result

        print(f"{key}: {result.shape[0]} rows x {result.shape[1]} columns")

        display(result.head(10))

Q1_1: 30 rows x 2 columns


,customer_id,total_spend
0,C0693,13776.55
1,C0082,12317.70
2,C0743,12266.85
3,C0525,12241.30
4,C0291,12150.70
5,C0267,12150.25
6,C0488,11825.50
7,C0699,11693.90
8,C0512,11617.65
9,C0332,10779.50


Q1_2: 30 rows x 3 columns


,city,category,revenue
0,Bengaluru,Electronics,233909.45
1,Bengaluru,Grocery,110915.65
2,Bengaluru,Dairy,87075.05
3,Bengaluru,Snacks,71549.40
4,Bengaluru,Personal Care,20193.85
5,Chennai,Electronics,195508.80
6,Chennai,Grocery,93817.45
7,Chennai,Dairy,78301.80
8,Chennai,Snacks,68325.70
9,Chennai,Personal Care,27675.40


Q1_3: 11 rows x 4 columns


,product_id,product_name,category,price_inr
0,P038,Bluetooth Speaker,Electronics,1999.0
1,P033,Wireless Earbuds,Electronics,1499.0
2,P034,Power Bank 10000mAh,Electronics,1299.0
3,P036,Phone Charger 20W,Electronics,799.0
4,P037,Smart Bulb,Electronics,599.0
5,P040,Memory Card 64GB,Electronics,549.0
6,P032,Sunscreen 50g,Personal Care,399.0
7,P030,Body Lotion 400ml,Personal Care,350.0
8,P039,Phone Stand,Electronics,349.0
9,P014,Ghee 500ml,Dairy,340.0


... (4 outputs left out)

Q1_6: 1175 rows x 6 columns


,city,customer_id,total_spend,rnk,dense_rnk,row_num
0,Bengaluru,C0082,12856.20,1,1,1
1,Bengaluru,C0525,12719.70,2,2,2
2,Bengaluru,C0699,12471.80,3,3,3
3,Bengaluru,C0543,11195.10,4,4,4
4,Bengaluru,C1147,10156.90,5,5,5
5,Bengaluru,C0407,9945.85,6,6,6
6,Bengaluru,C0942,8643.60,7,7,7
7,Bengaluru,C1126,8595.70,8,8,8
8,Bengaluru,C0916,8444.85,9,9,9
9,Bengaluru,C0098,8151.20,10,10,10


Q1_7: 183 rows x 3 columns


,order_date,daily_rev,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
5,2025-04-06,26323.80,121384.30
6,2025-04-07,15606.90,136991.20
7,2025-04-08,17055.45,154046.65
8,2025-04-09,20403.35,174450.00
9,2025-04-10,15359.90,189809.90


## 4. Review query outputs
Use the query names and column headings to explain what each result means. SQL concepts used in this project include filtering, joins, aggregation, grouping, ranking/window functions and cumulative totals.

In [6]:
# Compact overview of all query results
overview = pd.DataFrame([
    {"query": name, "rows": len(frame), "columns": ", ".join(frame.columns)}
    for name, frame in sql_results.items()
])
display(overview)

,query,rows,columns
0,Q1_1,30,"customer_id, total_spend"
1,Q1_2,30,"city, category, revenue"
2,Q1_3,11,"product_id, product_name, category, price_inr"
3,Q1_4,25,"customer_id, city, signup_date"
4,Q1_5,550,"customer_id, total_spend, avg_benchmark"
5,Q1_6,1175,"city, customer_id, total_spend, rnk, dense_rnk..."
6,Q1_7,183,"order_date, daily_rev, cumulative_revenue"


## 5. Validate the project checkpoints
The original `src/run_sql.py` contains exact benchmark assertions for the seven queries. Running it is the authoritative check.

In [7]:
from src.run_sql import run_queries
run_queries()

================ EXECUTING SQL QUERIES ================

--- Q1 Results (Shape: 30 rows, 2 cols) ---
customer_id  total_spend
      C0693     13776.55
      C0082     12317.70
      C0743     12266.85
      C0525     12241.30
      C0291     12150.70

--- Q2 Results (Shape: 30 rows, 3 cols) ---
     city      category   revenue
Bengaluru   Electronics 233909.45
Bengaluru       Grocery 110915.65
Bengaluru         Dairy  87075.05
Bengaluru        Snacks  71549.40
Bengaluru Personal Care  20193.85

--- Q3 Results (Shape: 11 rows, 4 cols) ---
product_id        product_name    category  price_inr
      P038   Bluetooth Speaker Electronics     1999.0
      P033    Wireless Earbuds Electronics     1499.0
      P034 Power Bank 10000mAh Electronics     1299.0
      P036   Phone Charger 20W Electronics      799.0
      P037          Smart Bulb Electronics      599.0

--- Q4 Results (Shape: 25 rows, 3 cols) ---
customer_id      city signup_date
      C0029 Hyderabad  2024-03-21
      C0037      P

{'Q1':    customer_id  total_spend
 0        C0693     13776.55
 1        C0082     12317.70
 2        C0743     12266.85
 3        C0525     12241.30
 4        C0291     12150.70
 5        C0267     12150.25
 6        C0488     11825.50
 7        C0699     11693.90
 8        C0512     11617.65
 9        C0332     10779.50
 10       C0543     10636.10
 11       C0775     10263.55
 12       C1024     10160.10
 13       C0426      9884.60
 14       C0308      9872.40
 15       C1147      9740.80
 16       C0407      9451.75
 17       C0930      9194.00
 18       C0130      9170.40
 19       C0147      9141.50
 20       C1146      8993.05
 21       C1136      8983.75
 22       C0696      8941.75
 23       C0167      8718.55
 24       C0819      8695.30
 25       C0942      8643.60
 26       C1000      8462.20
 27       C0916      8395.35
 28       C0454      8299.15
 29       C0424      8294.20,
 'Q2':          city       category    revenue
 0   Bengaluru    Electronics  233909.45
 1   B

## 6. Close the database connection

In [8]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.


## 7. Interpretation
For your capstone explanation, describe one business question answered by each SQL query and why SQL is useful alongside Pandas. **Part 2 complete:** the seven supplied SQL queries have been executed and their benchmark checks run.

In [9]:
import importlib
import src.run_sql

importlib.reload(src.run_sql)

from src.run_sql import run_queries

run_queries()

================ EXECUTING SQL QUERIES ================

--- Q1 Results (Shape: 30 rows, 2 cols) ---
customer_id  total_spend
      C0693     13776.55
      C0082     12317.70
      C0743     12266.85
      C0525     12241.30
      C0291     12150.70

--- Q2 Results (Shape: 30 rows, 3 cols) ---
     city      category   revenue
Bengaluru   Electronics 233909.45
Bengaluru       Grocery 110915.65
Bengaluru         Dairy  87075.05
Bengaluru        Snacks  71549.40
Bengaluru Personal Care  20193.85

--- Q3 Results (Shape: 11 rows, 4 cols) ---
product_id        product_name    category  price_inr
      P038   Bluetooth Speaker Electronics     1999.0
      P033    Wireless Earbuds Electronics     1499.0
      P034 Power Bank 10000mAh Electronics     1299.0
      P036   Phone Charger 20W Electronics      799.0
      P037          Smart Bulb Electronics      599.0

--- Q4 Results (Shape: 25 rows, 3 cols) ---
customer_id      city signup_date
      C0029 Hyderabad  2024-03-21
      C0037      P

{'Q1':    customer_id  total_spend
 0        C0693     13776.55
 1        C0082     12317.70
 2        C0743     12266.85
 3        C0525     12241.30
 4        C0291     12150.70
 5        C0267     12150.25
 6        C0488     11825.50
 7        C0699     11693.90
 8        C0512     11617.65
 9        C0332     10779.50
 10       C0543     10636.10
 11       C0775     10263.55
 12       C1024     10160.10
 13       C0426      9884.60
 14       C0308      9872.40
 15       C1147      9740.80
 16       C0407      9451.75
 17       C0930      9194.00
 18       C0130      9170.40
 19       C0147      9141.50
 20       C1146      8993.05
 21       C1136      8983.75
 22       C0696      8941.75
 23       C0167      8718.55
 24       C0819      8695.30
 25       C0942      8643.60
 26       C1000      8462.20
 27       C0916      8395.35
 28       C0454      8299.15
 29       C0424      8294.20,
 'Q2':          city       category    revenue
 0   Bengaluru    Electronics  233909.45
 1   B